# Step 1 — simulate

A later notebook: it does **not** take the varying values directly, it loads them
from `settings.json` (whose path nb2slurm passes in). It runs the Monte Carlo
estimate and saves the result under this job's `outdir`.

In [1]:
settings_path = "settings.json"

In [2]:
# Parameters
settings_path = "output/1000/3/settings.json"


In [3]:
import json
from pathlib import Path
import nb2slurm

# import the helper from scripts/ — works whether cwd is notebooks/ (local) or
# the project root (on SLURM).
try:
    from scripts.montecarlo import estimate_pi
except ImportError:
    import sys
    project_root = Path().resolve().parent
    sys.path.append(str(project_root))
    from scripts.montecarlo import estimate_pi

settings = nb2slurm.Settings.load(settings_path)
outdir = Path(settings["outdir"])

In [4]:
result_file = outdir / "result.json"

# idempotent: don't recompute if this job already finished
if result_file.exists():
    result = json.loads(result_file.read_text())
    print(f"reusing existing result: pi ~= {result['pi']}")
else:
    pi, inside, points = estimate_pi(settings["n_samples"], settings["seed"])
    result = {"pi": pi, "inside": inside, "n_samples": settings["n_samples"],
              "seed": settings["seed"]}
    result_file.write_text(json.dumps(result, indent=2))
    # save the points so the plot step can draw them
    (outdir / "points.json").write_text(json.dumps(points))
    print(f"n_samples={settings['n_samples']} seed={settings['seed']} -> pi ~= {pi}")

n_samples=1000 seed=3 -> pi ~= 3.168
